In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

import xgboost as xgb

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

I0000 00:00:1777357577.763249  131852 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777357580.655784  131852 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
file_path = "/home/home/code/xucenying/grid-intelligence/notebooks/susanta/consolidated_full.csv"
df=pd.read_csv(file_path)


In [3]:
df.shape

(265541, 26)

In [4]:
df.isnull().sum()

datetime_utc                             0
generation                             342
consumption                            359
wind_onshore                           343
temperature_c                        87896
humidity_percent                     87896
cloud_cover_percent                  87900
shortwave_radiation_wm2              87900
wind_speed_ms                        87896
ttf_gas                               6522
price                                  393
generation_renewable                   342
generation_non_renewable               342
wti_oil                                437
brent_oil                              437
natural_gas                            437
temperature_c_observed                 371
humidity_percent_observed              371
cloud_cover_percent_observed           371
shortwave_radiation_wm2_observed       371
wind_speed_ms_observed                 371
temperature_c_forecast              265178
humidity_percent_forecast           265178
cloud_cover

In [5]:
df.keys()

Index(['datetime_utc', 'generation', 'consumption', 'wind_onshore',
       'temperature_c', 'humidity_percent', 'cloud_cover_percent',
       'shortwave_radiation_wm2', 'wind_speed_ms', 'ttf_gas', 'price',
       'generation_renewable', 'generation_non_renewable', 'wti_oil',
       'brent_oil', 'natural_gas', 'temperature_c_observed',
       'humidity_percent_observed', 'cloud_cover_percent_observed',
       'shortwave_radiation_wm2_observed', 'wind_speed_ms_observed',
       'temperature_c_forecast', 'humidity_percent_forecast',
       'cloud_cover_percent_forecast', 'shortwave_radiation_wm2_forecast',
       'wind_speed_ms_forecast'],
      dtype='object')

In [6]:
data = df.drop(columns = ['temperature_c', 'humidity_percent', 'cloud_cover_percent',
       'shortwave_radiation_wm2', 'wind_speed_ms', 'temperature_c_forecast', 'humidity_percent_forecast',
       'cloud_cover_percent_forecast', 'shortwave_radiation_wm2_forecast',
       'wind_speed_ms_forecast'])

In [7]:
data.head(5)

,datetime_utc,generation,consumption,wind_onshore,ttf_gas,price,generation_renewable,generation_non_renewable,wti_oil,brent_oil,natural_gas,temperature_c_observed,humidity_percent_observed,cloud_cover_percent_observed,shortwave_radiation_wm2_observed,wind_speed_ms_observed
0,2018-09-30 22:00:00+00:00,51434.81,NaN,4292.89,NaN,49.30,11863.35,28945.23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-09-30 22:15:00+00:00,52085.57,NaN,4239.07,NaN,44.38,11809.11,29001.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2018-09-30 22:30:00+00:00,52345.33,NaN,4208.44,NaN,36.99,11632.71,29136.32,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2018-09-30 22:45:00+00:00,52902.03,NaN,4153.52,NaN,35.54,11937.48,28929.82,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2018-09-30 23:00:00+00:00,52799.83,42354.46,4047.26,NaN,46.50,12005.45,28880.65,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
data.keys()

Index(['datetime_utc', 'generation', 'consumption', 'wind_onshore', 'ttf_gas',
       'price', 'generation_renewable', 'generation_non_renewable', 'wti_oil',
       'brent_oil', 'natural_gas', 'temperature_c_observed',
       'humidity_percent_observed', 'cloud_cover_percent_observed',
       'shortwave_radiation_wm2_observed', 'wind_speed_ms_observed'],
      dtype='object')

 # ***Selecting data only from last one year***

In [9]:
data["datetime_utc"] = pd.to_datetime(data["datetime_utc"], utc=True)

start_date = pd.Timestamp("2025-02-01 00:00:00", tz="UTC")
end_date = pd.Timestamp("2026-02-01 00:00:00", tz="UTC")

data = data[
    (data["datetime_utc"] >= start_date) &
    (data["datetime_utc"] <= end_date)
]

In [10]:
data.tail()

,datetime_utc,generation,consumption,wind_onshore,ttf_gas,price,generation_renewable,generation_non_renewable,wti_oil,brent_oil,natural_gas,temperature_c_observed,humidity_percent_observed,cloud_cover_percent_observed,shortwave_radiation_wm2_observed,wind_speed_ms_observed
257284,2026-01-31 23:00:00+00:00,55732.36233,51641.31842,14644.42952,39.285,110.19,26734.05528,28722.69065,65.209999,70.690002,4.354,-2.400,97.079660,100.0,0.0,9.906544
257285,2026-01-31 23:15:00+00:00,55150.03397,50830.25718,14331.58355,39.285,95.14,26608.13671,28119.43306,65.209999,70.690002,4.354,-2.525,96.717320,100.0,0.0,9.199272
257286,2026-01-31 23:30:00+00:00,54411.85230,50287.21018,14160.32318,39.285,104.76,26335.74264,27619.46746,65.209999,70.690002,4.354,-2.650,96.354996,100.0,0.0,8.492001
257287,2026-01-31 23:45:00+00:00,54086.24809,49758.64134,13969.78983,39.285,95.89,26146.36708,27237.10131,65.209999,70.690002,4.354,-2.775,95.992660,100.0,0.0,7.784729
257288,2026-02-01 00:00:00+00:00,53869.80352,49125.44241,13968.65700,39.285,100.20,26089.38712,27100.08260,65.209999,70.690002,4.354,-2.900,95.630325,100.0,0.0,7.077457


# Preprocessing and cleaning #


1. Market data → step-like → ffill
2. Weather → smooth physics → interpolation
3. Observed weather → high-trust → preserve with ffill
4. Generation/consumption → continuous signals → interpolate
5. Solar → respects day/night physics
6. Target (price) → never fabricate

In [11]:
# Ensuring datetime as index
data['datetime_utc'] = pd.to_datetime(data['datetime_utc'])
data = data.set_index('datetime_utc').sort_index()

# -----------------------------
# 1. Defining column groups
# -----------------------------
weather_cols = [
    'temperature_c_observed', 'humidity_percent_observed',
    'cloud_cover_percent_observed',
    'shortwave_radiation_wm2_observed', 'wind_speed_ms_observed'
]

market_cols = [
    'ttf_gas', 'wti_oil', 'brent_oil', 'natural_gas'
]

core_cols = ['generation', 'consumption']
mix_cols = ['generation_renewable', 'generation_non_renewable']

# -----------------------------
# 2. Market variables → forward fill
# -----------------------------
data[market_cols] = data[market_cols].ffill()

# -----------------------------
# 3. Weather variables → time interpolation
# -----------------------------
data[weather_cols] = data[weather_cols].ffill().interpolate(method='time')

# -----------------------------
# 4. Core system variables → interpolate + ffill
# -----------------------------
data[core_cols] = data[core_cols].interpolate().ffill()

# -----------------------------
# 5. Energy mix → interpolate
# -----------------------------
data[mix_cols] = data[mix_cols].interpolate()

# -----------------------------
# 6. Wind generation → interpolate + light smoothing
# -----------------------------
data['wind_onshore'] = (
    data['wind_onshore']
    .interpolate()
    .rolling(3, min_periods=1)
    .mean()
)

# -----------------------------
# 7. Solar radiation (physics-aware)
# -----------------------------
# Nighttime → 0
night_mask = (data.index.hour < 6) | (data.index.hour > 20)
data.loc[night_mask, 'shortwave_radiation_wm2_observed'] = \
    data.loc[night_mask, 'shortwave_radiation_wm2_observed'].fillna(0)

# Daytime → interpolate
data['shortwave_radiation_wm2_observed'] = data['shortwave_radiation_wm2_observed'].interpolate()

# -----------------------------
# 8. Target variable →  NOT IMPUTING ANY VALUES
# -----------------------------
data = data[data['price'].notna()]

# -----------------------------
# 9. Final fallback (rare cases)
# -----------------------------
data = data.fillna(data.mean())

# -----------------------------
# 10. Making data for 15 mins interval
# -----------------------------
data = data.asfreq('15min')

In [12]:
data.head()

,generation,consumption,wind_onshore,ttf_gas,price,generation_renewable,generation_non_renewable,wti_oil,brent_oil,natural_gas,temperature_c_observed,humidity_percent_observed,cloud_cover_percent_observed,shortwave_radiation_wm2_observed,wind_speed_ms_observed
datetime_utc,,,,,,,,,,,,,,,
2025-02-01 00:00:00+00:00,43647.56,49590.034,6718.130000,53.237,145.16,13764.33,28316.47,72.529999,76.760002,3.044,-4.2,93.088250,0.00,0.0,3.244996
2025-02-01 00:15:00+00:00,42999.59,49284.771,6684.440000,53.237,130.90,13693.32,28107.40,72.529999,76.760002,3.044,-4.0,92.665380,0.75,0.0,3.204023
2025-02-01 00:30:00+00:00,42356.21,48936.994,6634.916667,53.237,126.30,13590.71,27926.22,72.529999,76.760002,3.044,-3.8,92.242510,1.50,0.0,3.163050
2025-02-01 00:45:00+00:00,42885.89,48774.808,6519.993333,53.237,120.70,13451.72,27967.99,72.529999,76.760002,3.044,-3.6,91.819640,2.25,0.0,3.122077
2025-02-01 01:00:00+00:00,42428.46,48611.645,6402.266667,53.237,131.50,13272.07,27606.37,72.529999,76.760002,3.044,-3.4,91.396774,3.00,0.0,3.081104


In [13]:
data.isnull().sum()

generation                          0
consumption                         0
wind_onshore                        0
ttf_gas                             0
price                               0
generation_renewable                0
generation_non_renewable            0
wti_oil                             0
brent_oil                           0
natural_gas                         0
temperature_c_observed              0
humidity_percent_observed           0
cloud_cover_percent_observed        0
shortwave_radiation_wm2_observed    0
wind_speed_ms_observed              0
dtype: int64

In [14]:
data.index.to_series().diff().value_counts()

datetime_utc
0 days 00:15:00    35040
Name: count, dtype: int64

# Feature engineering

In [15]:
##creating lags
lags = [1,4,8,24,96,192,672,1344]
for lag in lags:
    data[f"price_lag_{lag}"] = data["price"].shift(lag)

In [16]:
##creating rolling windows by mean:
windows = [1, 4, 8, 24, 96, 192, 672, 1344]

for wm in windows:
    data[f'price_roll_mean_{wm}'] = (
        data['price']
        .shift(1)              # IMPORTANT: avoid leakage
        .rolling(wm)
        .mean()
    )

In [17]:
##creating rolling windows by standard:
windows = [1, 4, 8, 24, 96, 192, 672, 1344]

for ws in windows:
    data[f'price_roll_std_{ws}'] = (
        data['price']
        .shift(1)              
        .rolling(ws)
        .std()
    )

# Creating multi-step target

In [18]:
horizon = 288  # 3 days

# Create 288 future targets
target_cols = []
for h in range(1, horizon + 1):
    col = f'target_t+{h}'
    data[col] = data['price'].shift(-h)
    target_cols.append(col)

# Drop NaNs
data = data.dropna(subset=target_cols)

/tmp/ipykernel_131852/720073495.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[col] = data['price'].shift(-h)
/tmp/ipykernel_131852/720073495.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[col] = data['price'].shift(-h)
/tmp/ipykernel_131852/720073495.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newf

In [19]:
data.head()

,generation,consumption,wind_onshore,ttf_gas,price,generation_renewable,generation_non_renewable,wti_oil,brent_oil,natural_gas,...,target_t+279,target_t+280,target_t+281,target_t+282,target_t+283,target_t+284,target_t+285,target_t+286,target_t+287,target_t+288
datetime_utc,,,,,,,,,,,,,,,,,,,,,
2025-02-01 00:00:00+00:00,43647.56,49590.034,6718.130000,53.237,145.16,13764.33,28316.47,72.529999,76.760002,3.044,...,130.9,157.7,145.3,133.5,120.30,153.20,145.20,122.60,122.52,145.10
2025-02-01 00:15:00+00:00,42999.59,49284.771,6684.440000,53.237,130.90,13693.32,28107.40,72.529999,76.760002,3.044,...,157.7,145.3,133.5,120.3,153.20,145.20,122.60,122.52,145.10,126.90
2025-02-01 00:30:00+00:00,42356.21,48936.994,6634.916667,53.237,126.30,13590.71,27926.22,72.529999,76.760002,3.044,...,145.3,133.5,120.3,153.2,145.20,122.60,122.52,145.10,126.90,129.26
2025-02-01 00:45:00+00:00,42885.89,48774.808,6519.993333,53.237,120.70,13451.72,27967.99,72.529999,76.760002,3.044,...,133.5,120.3,153.2,145.2,122.60,122.52,145.10,126.90,129.26,125.40
2025-02-01 01:00:00+00:00,42428.46,48611.645,6402.266667,53.237,131.50,13272.07,27606.37,72.529999,76.760002,3.044,...,120.3,153.2,145.2,122.6,122.52,145.10,126.90,129.26,125.40,130.34


**Training XG Boost model and getting predictions from it**

In [20]:
split = int(len(data) * 0.8)

train = data.iloc[:split]
test  = data.iloc[split:]

X_train = train.drop(columns=['price'] + target_cols)
y_train = train[target_cols]

X_test = test.drop(columns=['price'] + target_cols)
y_test = test[target_cols]

In [30]:
model_xgb = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model_xgb.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [57]:
xgb_pred = model_xgb.predict(X_test)

mae_xgb = mean_absolute_error(y_test, y_pred)
print("Overall MAE (3-day trajectory):", mae_xgb)

Overall MAE (3-day trajectory): 31.16868782043457


**Training LSTM model and getting predictions from it**

In [24]:
scaler = StandardScaler()

train_price = scaler.fit_transform(train[['price']])
test_price  = scaler.transform(test[['price']])

In [25]:
def create_sequences(data, window=96, horizon=288):
    X, y = [], []
    for i in range(len(data) - window - horizon):
        X.append(data[i:i+window])
        y.append(data[i+window:i+window+horizon])
    return np.array(X), np.array(y)

window = 96

X_train_seq, y_train_seq = create_sequences(train_price, window, horizon)
X_test_seq,  y_test_seq  = create_sequences(test_price,  window, horizon)

In [26]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(window, 1)),
    LSTM(32),
    Dense(horizon)
])

model.compile(optimizer='adam', loss='mse')

model.fit(
    X_train_seq, y_train_seq,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

Epoch 1/10


E0000 00:00:1777357919.972336  131852 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/home/.pyenv/versions/3.10.6/envs/grid-intelligence/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


386/386 ━━━━━━━━━━━━━━━━━━━━ 28s 68ms/step - loss: 0.6644 - val_loss: 0.5438
Epoch 2/10
386/386 ━━━━━━━━━━━━━━━━━━━━ 28s 72ms/step - loss: 0.5168 - val_loss: 0.5789
Epoch 3/10
386/386 ━━━━━━━━━━━━━━━━━━━━ 29s 76ms/step - loss: 0.4914 - val_loss: 0.6081
Epoch 4/10
386/386 ━━━━━━━━━━━━━━━━━━━━ 29s 74ms/step - loss: 0.4729 - val_loss: 0.6233
Epoch 5/10
386/386 ━━━━━━━━━━━━━━━━━━━━ 29s 75ms/step - loss: 0.4562 - val_loss: 0.6081
Epoch 6/10
386/386 ━━━━━━━━━━━━━━━━━━━━ 29s 75ms/step - loss: 0.4434 - val_loss: 0.6266
Epoch 7/10
386/386 ━━━━━━━━━━━━━━━━━━━━ 29s 74ms/step - loss: 0.4304 - val_loss: 0.7632
Epoch 8/10
386/386 ━━━━━━━━━━━━━━━━━━━━ 27s 70ms/step - loss: 0.4236 - val_loss: 0.7536
Epoch 9/10
386/386 ━━━━━━━━━━━━━━━━━━━━ 29s 76ms/step - loss: 0.4117 - val_loss: 0.7035
Epoch 10/10
386/386 ━━━━━━━━━━━━━━━━━━━━ 29s 76ms/step - loss: 0.3993 - val_loss: 0.7601


In [27]:
lstm_pred = model.predict(X_test_seq)

206/206 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step


In [58]:
mae_xgb = mean_absolute_error(y_test, y_pred)

mae_lstm = mean_absolute_error(y_true, lstm_pred)

print("Overall XGB MAE (3-day trajectory):", mae_xgb)

print("Overall LSTM MAE (3-day trajectory):", mae_lstm)


Overall XGB MAE (3-day trajectory): 31.16868782043457
Overall LSTM MAE (3-day trajectory): 38.91237571946342


# Comments

The ensemble method for 3-day trajectory prediction was not considered further, as both models exhibited MAE values exceeding 30, underperforming relative to prior benchmarks.